# Keep the latest record per customer

**Track:** sql | **Difficulty:** easy | **Tags:** deduplication, window functions, nulls

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("002-latest-record-per-customer")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "customer_updates": """
UEFSMRUEFYAFFdoCTBVQFQASAADAAgTLAAkBAOUJBwQA4Q0IANMNCADrDQgA1g0IAOMNCADeDQgA7g0IAO8NCADRDQgA6A0IAOoN
CADNDQgA2Q0IAPANCADXDQgA5w0IAMwNCADSDQgA2w0IAM4NCADiDQgAyg0IANwNCADYDQgA3Q0IAOANCADpDQgA7A0IAOYNCADP
DQgA2g0IANANCADtDQgA3w0IAMkNCADVDQg81AAAAAAAAADkAAAAAAAAABUAFcYBFcwBLBXwARUQFQYVBhwYCPAAAAAAAAAAGAjJ
AAAAAAAAABYAKAjwAAAAAAAAABgIyQAAAAAAAAAREQAAAGPwYgMAAADwAQEGH0AgDMRQGMJRHAWSKIvANE7wQNEiDdPiUENlDdeE
UYiDOUpmBVGDMcKhbZyjdN7HcdCRFREHhkEGHEQjXmPUUQoXhRAJlCDXSKamIWfAQNFgFWV4LMhnht0XFBUEFcYVFewFTBVyFQAS
AADjClgTAAAAdXNlcjIwM0BleGFtcGxlLmNvbRUXBDI5LhcAABcRLhQyNS5uZXdSGwAEMTE+GwAVTQQzNVYyAAA0YjIABDI3LjIA
FUkAMjJJABkXADJSLgAEMTQuFwAVxAQzOGJ3AAAzVvYABDA5PjIAGU0AMj4bABWWADMyfwAVFwAwVsQAADEy8gAVYAAyQnsAFUkE
NDBS3wAAMUKjARUyBDMxLjIAGRdWPwEAMFbAAAAxVkUAADFajgAAN2IgAQAxMm0BFY4EMDZSpQAAMlYXAAAwVrwAADIyjgAVXAwz
OEBlSu0CADAyigA1fAQxNj68ABUbADNC7QIVGwAwQoABFRsAMmYtAgAyVlYBADIyhAEVSQQzMz6aABnjVigBADNWEQEAMDJ2AhVF
ADFWRQAAMTI/ARWOADBqUQNWYAAAMlZ1BAAzVhcAADAydwAZpVZJAwAwVkoBADFWXAAAMTJAAhXTADJC4AEVdwAyMo4AFTIAM0It
AhUAFcYBFcwBLBXwARUQFQYVBhw2ACgTdXNlcjI0MEBleGFtcGxlLmNvbRgTdXNlcjIwMUBleGFtcGxlLmNvbRERAAAAY/BiAwAA
APABAQYfQCAMxFAYB5IghbIwjeA8UCRNVGVdWJNpw8YNHeZpH+RlYMhF4ug5B1maJ8SgqXqeE7kW1Mm2wQqQxON2r4hqICzG8gzQ
LDxadWl/d+DI4sUldExWn1rHuBoVFQQV0AEVygFMFRQVABIAAGhwBwAAAENoZW5uYWkGAAAASmFpcHVyBAAAAFB1bmUBHTxLb2xr
YXRhBQAAAERlbGhpAQkMS29jaAUvSE11bWJhaQkAAABIeWRlcmFiYWQBRkhJbmRvcmUJAAAAQmVuZ2FsdXJ1FQAVlAEVmgEsFfAB
FRAVBhUGHDYEKARQdW5lGAlCZW5nYWx1cnUREQAAAErwSQgAAAAiAQX+964BAQQfEAAgQxVBE2ZEdCgyIoQIQjNzaWR3JIKYUQCY
dZRXkCl0dwUzAgeAEkSBKGVViFlRkDBpgwMoAJlhhSkAFQQV4AwVqAtMFcwBFQASAACwBjAAKPjiYSsGAABDBGWaAQhMDVq8bC0G
AABszeE1MAYAAGKgXbkBGDA0M6IrLgYAAN5ZGacuAQgovFUKLAYAAOleiJMBCAzPcXTaATgMuTs2zwEQDPLBs+wBCAzDKvKSAQgM
bQnJ9AE4DBYWxTQBCAx3GBRGAQgMaWWG/gE4DLCpohwBKAxqcgVkAQgMD5FBngEgDFdkBHUBCAxoj2pdAYgM7/3riwEIDDpVWKgB
OAwoOxfLATAM4B4CFgEoDMd+i90BEAwDDuK6ASAMJRKvfQEQLCjPpqQvBgAAqpRh4gFADGqLxLcBIAzC57MGASAM8BdnxwEIDL/y
plYBCAxIzuVJAQgMYO6OlQEIDCx4VUsBMAhaPcMFaAz/iuAWARAMvvhBPAF4DIIJPrYBCAzwySGlARgMVSiM/wE4CDygIgVQDBAi
xUwBIAzbSpveARgMesk6yAGIDFFhK8EBMAyDBFWVAQgMdS4icgEYDJXpD6QhgAzBObcYAbgMu13EvwEIDJ65XegBSAz6NYvkAUgM
5OsHXQEQDBMrF3kBOAz3XX4HARAMDkyfiAEgDNRQ7MoBCAyOvg0aASAMFdy6awFoDC44S3YBGAyMwsHmAVgMJfZgbwEQCJ7NoiWA
DM6m8IUBECyL6Y/3KgYAAJTlttoBCAzJaajiARgMtKlRPAEIDFW/IwABWAz/HTeYAQgMdrwWZgEYDAK6ZtkBMAy3Mxn3ARgMsZGf
sAEQDNmfnPQBEAzsWIiHASgMi+vzgAEIDNkd/DQBCAzmhcnxAQgMc1fRfAEILIFNky0uBgAAyLE+2QE4DHGCXX0BCAzJxsbQASAM
cUwl6gEIDKEV6HoBGAyd/t2QAQgMxcsK9QEYDI0bIuUBEAxO3YF7AQgMwNFvJwEIDO3Ob+wBIAyC1eIYAWAMU2CNQgEYDEaS9GcB
EAykpVQ+ASA48NuJKy0GAAAIWgMqLgYAFQAV9gEV/AEsFfABFRAVBhUGHBgIAJXpD6QwBgAYCACxkZ+wKgYAFgooCACV6Q+kMAYA
GAgAsZGfsCoGABERAAAAe/B6DAAAABwBBf7+KAEFvvdsAQcfgIBgQCgYDoiEYmFgNBwPSEQyoVQsF0xGs+F0PJ8BKCTijEgl0wn1
SalWEFbL9ZLAYrIZrWa74XK6Ha/n+wGDwiGxaDwik0rVktn8OZ/QVVQ6pVatvSs2q91yu0ugt/oFh8VjchkAAAAAFQQVgA4VtgxM
FeABFQASAACABzAAb23LYisGAAAjddeaAQhMUI4lbi0GAACaCgo5MAYAAFA417kBGCwGfQ0sLgYAAHm5LKgBCCwvGMYLLAYAAO8D
kZQBCAw/qq3aATgMDSZhzwEQDAGbf+0BCAgw1HAFICzjHW66KgYAAOPmCvYBQAxOsuE0AQgMpoBtRgEIDN1l0gEBCAysaCMdATgM
seftZAEIDOoOcJ4BGAiZGcsFQAy/Gj92ARAMlGgJXwGYDHw2+IwBCAzozrapAYAMHHiZzAE4DPM4TRYBKAz/GqjdARAMJQH5uwEg
DGhGGH8BECwAvhqmLwYAAMyHeOMBQAwOH5u4ASAM4fheNgH4DGXJxQYBKAxT213IAQgMbWwFWAEIDH7vHX4BCAzLIGpKAQgMVCsR
lwEIDCNEkksBQCyq5m4XLgYAABKlKxcBEAzJkI49ARAMPjm5uQEQDHZGwLcBEAzDxVGmARAMqRK3/wFADOPyzXMBEAzodov/AQgM
JvccVwEYDM9R5UwBMAwCMfbfARAMSNImyQGwDMGPBc0BqAyO8IvBATAMkStclQEIDGi533IBIAx+P8cpARAMAeHJpAEoDKJc7hkB
8AwjFP/AAQgMsoVt6QFYDOPawOQBWAybmSleARAM6mfGeQFADLaNngcBEAw0gDWJASAMXUh5zAEIDDaTYxoBIAwo9gVsAWAMJrZM
dwEYDKCO0ecBWAx/hZRwARAMTEcBZgEIDBUc2YYBCCzf07r3KgYAAGYvItsBCAxMvCzjARgMdT37PQEIDOzdtwEBWAw9X1yZAQgI
idZhBUAM1bWW2gEwDFvH7/cBGAyRAhKxARAM0lK1egE4DNpRYfUBGAgmWS4FqAxYQhuBARgM1dx8NQEIDMvpf/IBCAx0CZZ9AQgM
BVLcLgHYLF/Q0torBgAAoOq2fQEIDEPlh9EBIAziNiPrAQgMj61hewEYDOi0RZEBCAwHTq/1ARgMC3ti5QEQDLGgeHwBCAzKt/cn
AQgMXrlt7QEgDIfIJhkBYAwgt7RCARgMxaP5aAEQDBTejT4BIDj/tFUsLQYAAJFRkCsuBgAVABXkARXqASwV8AEVEBUGFQYcGAgA
wY8FzTAGABgIAJECErEqBgAWACgIAMGPBc0wBgAYCACRAhKxKgYAEREAAABy8HEDAAAA8AEBBx+AgGBAKBgOiIRiYWA0HA9IRDKh
VCwXTEaz4XQ8H1BINCKVTCdUSrVitVwvWEw2o9Vstwkup9vxer4fMCgcEovGIzKpXDKbzic0Kp2+qFXrFZvVwrbcrvcLNobFY3LZ
fKYm0d+0es1uu98VBBlsNQAYBnNjaGVtYRUKABUEJQIYC2N1c3RvbWVyX2lkABUMJQIYBWVtYWlsJQBMHAAAABUMJQIYBGNpdHkl
AEwcAAAAFQQlAhgKdXBkYXRlZF9hdCUUTIwSHCwAAAAAABUEJQIYC2luZ2VzdGVkX2F0JRRMjBIcLAAAAAAAFvABGRwZXCYAHBUE
GTUABhAZGAtjdXN0b21lcl9pZBUCFvABFuoHFsoFJoIDJggcGAjwAAAAAAAAABgIyQAAAAAAAAAWACgI8AAAAAAAAAAYCMkAAAAA
AAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAPABAAAAJgAcFQwZNQAGEBkYBWVtYWlsFQIW8AEWtBgW4Agm3gsm0gUcNgAoE3Vz
ZXIyNDBAZXhhbXBsZS5jb20YE3VzZXIyMDFAZXhhbXBsZS5jb20REQAZLBUEFQAVAgAVABUQFQIAPBaQJhkGGSYA8AEAAAAmABwV
DBk1AAYQGRgEY2l0eRUCFvABFtoDFtoDJpwQJrIOHDYEKARQdW5lGAlCZW5nYWx1cnUREQAZLBUEFQAVAgAVABUQFQIAPBbcCxkG
GSYE7AEAAAAmABwVBBk1AAYQGRgKdXBkYXRlZF9hdBUCFvABFvwPFsoOJtYdJowSHBgIAJXpD6QwBgAYCACxkZ+wKgYAFgooCACV
6Q+kMAYAGAgAsZGfsCoGABERABksFQQVABUCABUAFRAVAgA8KQYZJgrmAQAAACYAHBUEGTUABhAZGAtpbmdlc3RlZF9hdBUCFvAB
FooRFsYPJq4tJtYgHBgIAMGPBc0wBgAYCACRAhKxKgYAFgAoCADBjwXNMAYAGAgAkQISsSoGABERABksFQQVABUCABUAFRAVAgA8
KQYZJgDwAQAAABb+RBbwASYIFpQwABkcGAxBUlJPVzpzY2hlbWEYzAMvLy8vLzFBQkFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNn
QUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBVUFBQURnQUFBQW9BQUFBSFFBQUFBNEFBQUFCQUFBQUVULy8vOEFB
QUVLRUFBQUFCd0FBQUFFQUFBQUFBQUFBQXNBQUFCcGJtZGxjM1JsWkY5aGRBRE8vLy8vQUFBQ0FIVC8vLzhBQUFFS0VBQUFBQ1FB
QUFBRUFBQUFBQUFBQUFvQUFBQjFjR1JoZEdWa1gyRjBBQUFBQUFZQUNBQUdBQVlBQUFBQUFBSUFyUC8vL3dBQUFRVVFBQUFBR0FB
QUFBUUFBQUFBQUFBQUJBQUFBR05wZEhrQUFBQUEyUC8vLzlULy8vOEFBQUVGRUFBQUFCd0FBQUFFQUFBQUFBQUFBQVVBQUFCbGJX
RnBiQUFBQUFRQUJBQUVBQUFBRUFBVUFBZ0FCZ0FIQUF3QUFBQVFBQkFBQUFBQUFBRUNFQUFBQUNRQUFBQUVBQUFBQUFBQUFBc0FB
QUJqZFhOMGIyMWxjbDlwWkFBSUFBd0FDQUFIQUFnQUFBQUFBQUFCUUFBQUFBQUFBQUE9ABggcGFycXVldC1jcHAtYXJyb3cgdmVy
c2lvbiAyNS4wLjEZXBwAABwAABwAABwAABwAAADCBAAAUEFSMQ==
""",
}

_data_dir = pathlib.Path("data") / "002-latest-record-per-customer"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

customer_updates = spark.read.parquet((_data_dir / "customer_updates.parquet").resolve().as_posix())
customer_updates.createOrReplaceTempView("customer_updates")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'city': '8d1c3c71639d0daafbd4f2c8c814b7d1e6f27d514b6c1b167d201cf347bd02f7',
                         'customer_id': 'a1dffe6617706852d925f398839ca47446954d3e51c84ff58c2037488f1b609d',
                         'email': '18066f08ce819802c9e5fca6dabcde48003e89911d45ef681fbd7816e976691d',
                         'updated_at': '9ee15497d3281378d73e2e6c933ab106e1ee1d03ed9d4282a875077e0e545303'},
 'columns': [['city', 'string'],
             ['customer_id', 'bigint'],
             ['email', 'string'],
             ['updated_at', 'timestamp_ntz']],
 'fingerprint': 'cf2592181da197992373281b5d1b26c996d66e814a81f39b49c112637c91186e',
 'order_matters': False,
 'row_count': 40}

check = make_check(EXPECTED)

## The situation

A source system sends a row to `customer_updates` every time a customer record changes.
The downstream `customers` table must hold the current state: one row per customer.

The feed is not clean. The same row is sometimes delivered twice, some corrections
arrive with the same `updated_at` as the row they correct, and some rows have no
`updated_at` at all.

## Input tables

### `customer_updates`

One row per delivered change. A customer has one or more rows.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | The customer the change is for. |
| `email` | STRING | Email after the change. |
| `city` | STRING | City after the change. NULL means the customer has no city on record. |
| `updated_at` | TIMESTAMP_NTZ | When the change happened in the source system. Can be NULL. |
| `ingested_at` | TIMESTAMP_NTZ | When the row arrived in the feed. Never NULL. |

## Requirement

Return the current row for every customer.

- The current row is the one with the latest `updated_at`.
- If two rows of a customer have the same `updated_at`, the one with the later
  `ingested_at` wins. It is a correction of the other.
- A row with no `updated_at` is older than any row that has one. It is only the
  current row if the customer has no dated rows.
- A row delivered twice must not produce two output rows.
- Take all output columns from the same input row. If the current row has a NULL
  `city`, the output has a NULL `city`.

## Expected output

Exactly one row per customer.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | |
| `email` | STRING | From the current row. |
| `city` | STRING | From the current row. Can be NULL. |
| `updated_at` | TIMESTAMP_NTZ | From the current row. Can be NULL. |

In [ ]:
customer_updates.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `customer_updates`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)